# Cost Cup — Project Walkthrough (KMeans Archetypes + Transitions)

This notebook is the **methodology + reproducibility walkthrough** for the Cost Cup project.

It covers:
- the data pipeline (raw → features → archetypes → transitions)
- sanity checks we ran to validate the tables
- the **KMeans archetype** method (with the **two required visualizations**)
- the **transition probability** model (with Dirichlet smoothing)
- how to launch the Dash app locally


## 0) Setup

Run from the repo root so imports and relative paths work. The cell below auto-adjusts the working directory if you started in `notebooks/`.


In [1]:
import os, sys
from pathlib import Path

# Move to repo root if needed
cwd = Path.cwd()
if (cwd / "dash_app").exists():
    repo = cwd
elif (cwd.parent / "dash_app").exists():
    repo = cwd.parent
else:
    # last resort: walk up
    repo = next(p for p in cwd.parents if (p / "dash_app").exists())

os.chdir(repo)
print("Repo root:", Path.cwd())

# Choose DB config mode
# - aws: uses AWS_DB_* env vars (RDS)
# - local: uses ENDPOINT/DB_USER/... from your .env
os.environ.setdefault("APP_ENV", "aws")

print("APP_ENV =", os.getenv("APP_ENV"))


Repo root: /Users/ericwiniecke/Documents/github/cost_cup
APP_ENV = aws


## 1) Pipeline map

Raw → resolved identities → player-game (ES) → player-season → cleaning/capping → KMeans archetypes (F/D) → transitions + Dirichlet smoothing → Dash app

The repo is **SQL-first**: most transformations are SQL scripts / stored procs; Python orchestrates execution and runs KMeans.


## 2) Methods (math)

### KMeans archetypes

We represent each player-season as a feature vector $x_i$ and fit KMeans ($K=3$) within each position group (F/D):

$$\min_{\{\mu_k\}_{k=1}^K} \sum_{i=1}^{N} \lVert x_i - \mu_{z_i} \rVert^2$$

Inputs are standardized to avoid scale-dominance:

$$\tilde{x}_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

### Rate-based ES features

$$\text{CF60} = 3600 \cdot \frac{\text{CF}}{\text{TOI}_{ES}}, \quad \text{CA60} = 3600 \cdot \frac{\text{CA}}{\text{TOI}_{ES}}$$

### Smoothed transition probabilities (Dirichlet)

Let $n_{ij}$ be observed transitions from cluster $i \to j$. With Dirichlet prior $\alpha_{ij}$, the posterior mean is:

$$\mathbb{E}[p_{ij} \mid \mathbf{n}_i] = \frac{n_{ij}+\alpha_{ij}}{\sum_{j'} (n_{ij'}+\alpha_{ij'})}$$

This shrinkage stabilizes probabilities (avoids brittle 0%/100% under sparse data).


## 3) Connect to Postgres and load the modeling dataset

We load the *clean* player-season archetype feature table and filter to **train seasons** for the KMeans diagnostics.


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import text
from db_utils import get_db_engine

FEATURES = ['toi_per_game', 'es_share', 'g60', 'a60', 'p60', 's60', 'hit60', 'blk60', 'take60', 'give60', 'penl60', 'fo_win_pct', 'cf60', 'ca60', 'cf_percent']
TRAIN_SEASONS = [20182019, 20192020, 20202021, 20212022, 20222023]

def load_pos_df(pos: str, seasons=TRAIN_SEASONS) -> pd.DataFrame:
    pos = pos.upper()
    if pos not in ("F","D"):
        raise ValueError("pos must be 'F' or 'D'")
    eng = get_db_engine()
    try:
        sql = text("""

SELECT f.*, p.pos
FROM mart.player_season_archetype_features_modern_truth_clean f
JOIN mart.player_season_pos_modern p
  ON p.season=f.season AND p.player_id=f.player_id
WHERE p.pos = :pos AND f.season = ANY(:seasons)

""")
        df = pd.read_sql_query(sql, eng, params={"pos": pos, "seasons": list(seasons)})
    finally:
        eng.dispose()
    # Fill faceoff win% for players with no draws
    if "fo_win_pct" in df.columns:
        df["fo_win_pct"] = df["fo_win_pct"].fillna(0.5)
    df = df.dropna(subset=FEATURES).copy()
    return df

dfF = load_pos_df("F")
dfD = load_pos_df("D")
print("F train rows:", len(dfF), "players:", dfF["player_id"].nunique())
print("D train rows:", len(dfD), "players:", dfD["player_id"].nunique())


2026-03-03 21:57:21,864 - INFO - Using AWS database config (APP_ENV=aws). host=database-1.cp82gws4kooz.us-east-2.rds.amazonaws.com db=hockey_stats port=5432


## 4) Required KMeans visualization #1: Elbow + Silhouette (K = 2..8)

We compute **inertia** (elbow) and **silhouette score** (cluster separation) for a sweep of K values on the **train seasons only**, separately for F and D.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def k_sweep(df_train: pd.DataFrame, k_min=2, k_max=8, n_init=50, seed=42):
    X = df_train[FEATURES].to_numpy(float)
    Xs = StandardScaler().fit_transform(X)

    ks = list(range(k_min, k_max + 1))
    inertia = []
    sil = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=n_init, random_state=seed)
        labels = km.fit_predict(Xs)
        inertia.append(km.inertia_)
        sil.append(silhouette_score(Xs, labels))
    return ks, inertia, sil

def plot_k_sweep(pos: str):
    df = dfF if pos.upper()=="F" else dfD
    ks, inertia, sil = k_sweep(df)

    # Elbow
    plt.figure()
    plt.plot(ks, inertia, marker="o")
    plt.title(f"{pos.upper()}: KMeans elbow (train seasons only)")
    plt.xlabel("K")
    plt.ylabel("Inertia (within-cluster SSE)")
    plt.show()

    # Silhouette
    plt.figure()
    plt.plot(ks, sil, marker="o")
    plt.title(f"{pos.upper()}: Silhouette score (train seasons only)")
    plt.xlabel("K")
    plt.ylabel("Silhouette score")
    plt.show()

plot_k_sweep("F")
plot_k_sweep("D")


## 5) Required KMeans visualization #2: Cluster profiles (feature means by cluster)

We fit the final model with **K = 3** on train seasons, then plot the **mean feature values** per cluster.

This makes clusters interpretable (e.g., offensive/high-event vs defensive/low-event) and is the basis for our cluster labels.


In [ ]:
from sklearn.preprocessing import StandardScaler

K_FIXED = 3

def fit_kmeans_and_profile(df_train: pd.DataFrame, pos: str):
    X = df_train[FEATURES].to_numpy(float)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    km = KMeans(n_clusters=K_FIXED, n_init=50, random_state=42)
    labels = km.fit_predict(Xs)

    prof = df_train[FEATURES].copy()
    prof["cluster"] = labels
    means = prof.groupby("cluster")[FEATURES].mean()

    # Plot a small set of headline features (adjust as needed)
    headline = ["toi_per_game","cf60","ca60","cf_percent","p60","s60","hit60","blk60"]
    headline = [c for c in headline if c in means.columns]

    plt.figure(figsize=(10,4))
    for k in means.index:
        plt.plot(headline, means.loc[k, headline].values, marker="o", label=f"cluster {k}")
    plt.title(f"{pos.upper()}: cluster profile (means, train seasons)")
    plt.xticks(rotation=30, ha="right")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return km, scaler, means

kmF, scF, meansF = fit_kmeans_and_profile(dfF, "F")
kmD, scD, meansD = fit_kmeans_and_profile(dfD, "D")

meansF.head(), meansD.head()


## 6) Sanity checks we ran (examples)

These are the checks we used to confirm the **player-game truth** table is sound before modeling.

Run these in psql (or via `sql/sanity/modern/`).

```sql
-- Duplicates at intended grain
SELECT COUNT(*) AS rows, COUNT(DISTINCT (game_id, player_id)) AS distinct_gp
FROM mart.player_game_features_20242025_truth;

-- Null keys
SELECT
  COUNT(*) FILTER (WHERE game_id IS NULL)   AS game_id_null,
  COUNT(*) FILTER (WHERE player_id IS NULL) AS player_id_null,
  COUNT(*) FILTER (WHERE team_id IS NULL)   AS team_id_null
FROM mart.player_game_features_20242025_truth;

-- Referential integrity (team_id -> dim_team_code)
SELECT COUNT(*) FILTER (WHERE tc.team_id IS NULL) AS team_join_miss
FROM mart.player_game_features_20242025_truth f
LEFT JOIN dim.dim_team_code tc ON tc.team_id = f.team_id;
```


## 7) Edge case example: Extreme CA is often team environment (ANA)

We used a diagnostic query that ranks team-games by **team total CA** and shows the top-5 skaters by CA in those games.

This helps avoid interpreting single-game CA spikes as purely a player-quality signal.


In [ ]:
# Load the CSV you exported from psql (example file name)
import pandas as pd
from pathlib import Path

csv_path = Path("ana_high_ca.csv")
if csv_path.exists():
    ana = pd.read_csv(csv_path)
    display(ana.head(10))
else:
    print("ana_high_ca.csv not found in repo root. If you exported it elsewhere, copy it into the repo root or update csv_path.")


## 8) Launch the Dash app from a notebook (optional)

If you want to run the full Dash app while staying inside the notebook UI, launch it in a subprocess.


In [ ]:
import os, subprocess, sys, textwrap

env = os.environ.copy()
env.setdefault("APP_ENV", "aws")  # or "local"

proc = subprocess.Popen([sys.executable, "-m", "dash_app.app"], env=env)
print("Dash starting… open http://127.0.0.1:8050")
print("To stop later: proc.terminate()")
